# 03_Main_InitialClustering
Finalized and run june 2nd 2026

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
import tempfile
from muon import prot as pt
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

# Helper functions from Mimi

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata

def filter_high_var(adata, n):
    #sc.pp.highly_variable_genes(
    #adata,
    #flavor="seurat_v3",
    #n_top_genes=n,
    #layer="counts",
    #batch_key="pool",
    #subset=True,
    #span = 0.75) #default = 0.3. It's the fraction of cells used for estimation of hvg - if I have 150 cells in one sample but 150.000 cells in total then this sample becomes an issue. 
    
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)

    leiden_column_name = f"leiden_{resolution}"
    adata.obs[leiden_column_name] = adata_tmp.obs['leiden']

# Loading data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_raw.h5mu")


In [ ]:
mdata["gex"].obs["batch"].value_counts()

# Annotation - Compartment

In [ ]:
MyDict = {
    'Pt1': 'CSF',
    'Pt5': 'CSF',
    'Pt4': 'CSF',
    'Pt8': 'CSF',
    'Pt11': 'CSF',
    'Pt12': 'CSF',
    'Pt14': 'CSF',
    'Pt17': 'CSF',
    'Pt18': 'CSF',
    'Pt19': 'CSF',
    'Pt1_PBMC': 'PB',
    'Pt4_PBMC': 'PB',
    'Pt5_PBMC': 'PB',
    'Pt8_PBMC': 'PB',
    'Pt12_PBMC': 'PB',
    'Pt11_PBMC': 'PB',
    'Pt17_PBMC': 'PB',
    'Pt19_PBMC': 'PB',
    'Pt20_PBMC': 'PB',
    'Pt21_PBMC': 'PB',
    'Pt18_PBMC': 'PB',
    'Pt14_PBMC': 'PB',
    'Pt20': 'CSF',
    'Pt21': 'CSF'
}

## Create mdata["gex"].obs["Compartment"] by mapping the dictionary to mdata["gex"].obs["pool"]
mdata["gex"].obs["Compartment"] = (
    mdata["gex"].obs["pool"]
    .map(MyDict)
)



In [ ]:
mdata["gex"].obs["Compartment"].value_counts()

# Overclustering at leiden resolution = 4

## Re-normalization after merging
According to this https://www.sc-best-practices.org/cellular_structure/integration.html renormalization after merging is a good idea. For this you first overwrite .X with raw counts from the counts layer. 

In [ ]:
# I renormalize as it is recommended by sc-best-practices when merging data.
mdata["gex"].X = mdata["gex"].layers["counts"].copy()
sc.pp.normalize_total(mdata["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(mdata["gex"])

#### Leiden (over)clustering res = 4 was picked to adequately capture B cells and T-B cells interacting so to label them separately

In [ ]:
calculate_leiden(mdata["gex"], resolution=4, n_high_var=5000, remove_tcr_genes=True)   
calculate_umap(mdata["gex"], n_high_var=5000, remove_tcr_genes=True)

In [ ]:
sc.pl.umap(mdata["gex"], color=["leiden_4"], legend_loc="on data", legend_fontweight= 'bold', frameon= False , vmin=0, legend_fontoutline=2)
mu.pl.embedding(mdata, basis="gex:umap", color="airr:chain_pairing", size=5)
mu.pl.embedding(mdata, basis="gex:umap", color="gex:donor", size = 20)
mu.pl.embedding(mdata, basis="gex:umap", color="gex:CD19", groups = "two full chains", size = 20)

In [ ]:
sc.pl.umap(mdata["gex"], color=["Compartment"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, size=5)
sc.pl.umap(mdata["gex"], color=["CD4", "CD8A"], legend_fontweight= 'bold', frameon= False , vmin=0, legend_fontoutline=2, size=5)

### Check cell counts

In [ ]:
# Look at occurence of samples
pd.DataFrame(mdata["gex"].obs[['Compartment']].value_counts())

In [ ]:
mdata["gex"].obs["batch"].value_counts()

# Define catpat, i.e. Cohort category of Control and RRMS

In [ ]:
#Group annotation
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]

#Match the lists defined above to define "RRMS" and "Control"
mdata["gex"].obs["catpat"] = pd.Categorical(np.where(mdata["gex"].obs["donor"].isin(patient_list), "RRMS", np.where(mdata["gex"].obs["donor"].isin(control_list), "Control", "Error")))


# Define a clone handle

In [ ]:
# Define clone_handle - a combination of the donor information and the clone_id information to make each shared clone unique in each donor. 
mdata["gex"].obs["clone_handle"] = mdata["gex"].obs["donor"].astype(str) + "-" + mdata["gex"].obs["clone_id"].astype(str)

print(mdata["gex"].obs["clone_handle"].value_counts().head(5))

# If clone_ind is "NaN" then clone_handle should also be NaN
mdata["gex"].obs["clone_handle"] = np.where(mdata["gex"].obs["clone_id"].isna(), pd.NA, mdata["gex"].obs["clone_handle"])

# #Remove ".0" from the clone handle
# mdata["gex"].obs["clone_handle"] = (
#     mdata["gex"].obs["clone_handle"]
#     .astype(str)
#     .str.replace(r"\.0$", "", regex=True)  #\ escape     #.0 string to match    $ match end of string
# )

# If the edits worked, then na should not be the most counted (otherwise it will be)
print(mdata["gex"].obs["clone_handle"].value_counts().head(5))

# Manual clone count

In [ ]:
## Calculate clone_size = number of rows per clone_id
mdata["gex"].obs["clone_size"] = mdata["gex"].obs.groupby("clone_handle", observed=True)["clone_handle"].transform("count")

# Define NA clones as 0 clone count
mdata["gex"].obs["clone_size"] = np.where(mdata["gex"].obs["clone_handle"].isna(), 0, mdata["gex"].obs["clone_size"])

## Share labels with airr
mdata["airr"].obs["clone_size"] = mdata["gex"].obs["clone_size"].copy()

mdata["gex"].obs[["clone_handle","clone_size"]].sort_values("clone_size", ascending=False).drop_duplicates()

In [ ]:
## Check the consistency between clone_size and clone_id_size
mdata["gex"].obs["clone_id_size"] = mdata["airr"].obs["clone_id_size"].copy()

## Define the dataframe
plot_df = mdata["gex"].obs[["clone_id_size", "clone_size", "clone_handle"]].drop_duplicates()

## XY scatter plot with a regression line to check the consistency between clone_id_size and clone_size
### The two metrics are not fully conssitent, but this should be explained by the clone_ids being called togehter with the multimer dataset to have the same clone_ids across analysis
# from scipy.stats import pearsonr

# # Calculate correlation
# r, p = pearsonr(
#     plot_df["clone_id_size"],
#     plot_df["clone_size"]
# )

# # Plot
# fig, ax = plt.subplots(figsize=(4, 4))

# sb.regplot(
#     data=plot_df,
#     x="clone_id_size",
#     y="clone_size",
#     scatter_kws={"s": 30},
#     line_kws={"lw": 2},
#     ax=ax
# )

# ax.set_xlabel("clone_id_size")
# ax.set_ylabel("clone_size")

# ax.text(
#     0.05,
#     0.95,
#     f"r = {r:.3f}\np = {p:.2e}",
#     transform=ax.transAxes,
#     va="top"
# )

# sb.despine()
# plt.grid(False)
# plt.tight_layout()
# plt.show()

## 

In [ ]:
### Drop the clone_id_size to not be confused
mdata["gex"].obs = mdata["gex"].obs.drop(columns=["clone_id_size"])

# Post-clustering QC

### Clean up - remove ribosomal outliers cluster 58 and cluster 60

In [ ]:
mdata["gex"].var['ribo'] = mdata["gex"].var_names.str.startswith('RPS') | mdata["gex"].var_names.str.startswith('RPL')
sc.pp.calculate_qc_metrics(mdata["gex"], qc_vars=['ribo'], percent_top=None, log1p=False, inplace=True)

In [ ]:
plt.rcParams["figure.figsize"] = (4, 3)
sc.pl.umap(mdata["gex"], color=["pct_counts_ribo", "Compartment", "chain_pairing"], legend_fontweight='bold', color_map = 'RdYlBu_r', frameon= False , vmin=0, legend_fontoutline=2, size=10)

# Make the figure wider using figsize
with plt.rc_context({"figure.figsize": (7, 4)}):
    sc.pl.violin(
        mdata["gex"], 
        keys="pct_counts_ribo",
        groupby="leiden_4",
        multi_panel=False,
        rotation=45,
        show=False
    )

plt.xticks(rotation=45, ha='right')  # fix alignment under ticks
plt.tight_layout()                   # adjust spacing if needed
plt.show()

plt.rcParams["figure.figsize"] = (7, 6)
sc.pl.umap(mdata["gex"], color=["leiden_4"], groups=["58", "60"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=10)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
# Removing outlier clusters
mask = ~(mdata["gex"].obs["leiden_4"].isin(["58", "60"]))  ## "~" is a negation sign
mu.pp.filter_obs(mdata["gex"], mask)

mdata.update()

In [ ]:
plt.rcParams["figure.figsize"] = (7, 6)
sc.pl.umap(mdata["gex"], color=["leiden_4"], legend_fontweight='bold', frameon= False , vmin=0, legend_fontoutline=2, legend_loc = "on data", size=10)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

## Clean up - remove CITEseq negative cells from blood. 
There are cells from PB without CITEseq signal. They contribute to noise later. 

In [ ]:
# cd4cd8_cs_assignment was called for individual files in preprocessing
mdata["gex"].obs["cd4cd8_cs_assignment"] = mdata["prot"].obs["cd4cd8_cs_assignment"].copy()
mdata["gex"].obs[["Compartment", "cd4cd8_cs_assignment"]].value_counts()

In [ ]:
# Removing adt_negative f
mask = ~((mdata["gex"].obs["Compartment"] == "PB") & (mdata["gex"].obs["cd4cd8_cs_assignment"] == "adt_negative"))  #~ reverse the bolean statement. This is important, because you want only the ones you pick to be false
mu.pp.filter_obs(mdata["gex"], mask)
mdata.update()
mdata["gex"].obs[["Compartment", "cd4cd8_cs_assignment"]].value_counts()

# Redo clustering

In [ ]:
## Filter genes away that are present in very few cells
## Threshold from: https://muon-tutorials.readthedocs.io/en/latest/cite-seq/1-CITE-seq-PBMC-5k.html
mu.pp.filter_var(mdata["gex"], 'n_cells_by_counts', lambda x: x >= 3)
mdata.update()

In [ ]:
## Renormalize after subsetting on specific genes. 
mdata["gex"].X = mdata["gex"].layers["counts"].copy()
mdata["gex"].uns.pop("log1p", None) #Removes log1P if it exists
sc.pp.normalize_total(mdata["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(mdata["gex"])

In [ ]:
calculate_leiden(mdata["gex"], resolution=4.5, n_high_var=5000, remove_tcr_genes=True)   
calculate_umap(mdata["gex"], n_high_var=5000, remove_tcr_genes=True)

In [ ]:
sc.pl.umap(mdata["gex"], color=["leiden_4.5"], legend_loc="on data", legend_fontweight= 'bold', frameon= False , vmin=0, legend_fontoutline=2)
mu.pl.embedding(mdata, basis="gex:umap", color="airr:chain_pairing", size=5)
mu.pl.embedding(mdata, basis="gex:umap", color="prot:cd4cd8_cs_assignment", size=5)

# Export - after clustering

In [ ]:
mdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_clustered.h5mu")